# SMILES 与 RDKit：分五步运行

从上到下点击每个单元左侧的 ▶。第一次运行完成环境安装；以后在同一运行时重复第①步会检查并复用环境。

①准备环境 → ②设置参数 → ③调用SKILL.md → ④查看结果 → ⑤下载。修改半径为3后，重新运行②至⑤，每次真实调用保存到新目录。

Colab左侧钥匙Secret中，授权本Notebook读取你自己的`DEEPSEEK_API_KEY`。不要把密钥写进代码，不要上传.env。第③步会向DeepSeek发送本页教学结构、Skill指令和参数，并执行限定RDKit工具；仅看页面和下载不会调用模型。

完整模块位于配套`runtime`目录，Notebook携带同一ZIP作为附件，运行代码中没有源码长字符串。私有仓库访问仍使用Colab正常GitHub授权；准备步骤不读取GitHub Token。附件读取使用Colab内部消息接口，接口失败时会停止并明确提示。


In [ ]:
# @title ① 准备环境（首次安装，重复执行复用）
from pathlib import Path
import base64, hashlib, json, sys, tempfile
from google.colab import _message
document = _message.blocking_request("get_ipynb", timeout_sec=20)
if not document or "ipynb" not in document:
    raise RuntimeError("未取得本Notebook附件。请等待页面加载后重试本步骤；不要填写GitHub Token。")
notebook = document["ipynb"]
attachment = notebook["metadata"].get("adp_runtime_bundle")
if not attachment:
    raise RuntimeError("本副本缺少配套模块附件，请重新从本版本运行入口打开。")
payload = base64.b64decode(attachment["zip_base64"], validate=True)
assert hashlib.sha256(payload).hexdigest() == '9aa81e8912b81787ff338d9461e04dcb21204a72b1cc9731cead3755d0906a8a', "配套模块校验失败"
bundle = Path(tempfile.gettempdir()) / ("rdkit-runtime-" + '9aa81e8912b81787ff338d9461e04dcb21204a72b1cc9731cead3755d0906a8a'[:16] + ".zip")
bundle.write_bytes(payload)
sys.path.insert(0, str(bundle))
from colab_workflow import Lab
lab = Lab.prepare(bundle, notebook=notebook)


## ② 参数与输入

样本为14条教学结构；半径默认2，改成3可比较指纹差异。SMARTS查找含OH的羧酸片段；参照005是苯。结构和错误记录全部保留。


In [ ]:
# @title ② 设置分析参数
RADIUS = 2 # @param {type:"integer"}
FP_SIZE = 2048 # @param [1024,2048,4096] {type:"raw"}
INCLUDE_CHIRALITY = True # @param {type:"boolean"}
QUERY_SMARTS = "[CX3](=O)[OX2H1]" # @param {type:"string"}
REFERENCE_ID = "005" # @param {type:"string"}
parameters = dict(radius=RADIUS, fp_size=FP_SIZE, include_chirality=INCLUDE_CHIRALITY,
                  query_smarts=QUERY_SMARTS, reference_id=REFERENCE_ID)
print(json.dumps(parameters, ensure_ascii=False, indent=2))


## ③ 调用 Skill

下面明确读取并传入SKILL.md原文。配套模块负责凭据读取、子进程与超时清理；Agent根据同一Skill调用限定工具。每点击一次会发起一次真实模型与工具任务。


In [ ]:
# @title ③ 读取 SKILL.md 并提交一次真实调用
skill_file = lab.workspace / "SKILL.md"
skill_text = skill_file.read_text(encoding="utf-8")
print("SKILL.md SHA-256:", hashlib.sha256(skill_file.read_bytes()).hexdigest())
run_output = lab.call_skill(skill_text=skill_text, parameters=parameters)


## ④ 结果与解释

查看程序核验回执、分子结构图、Morgan相似度图，展开模型原文。文件状态以程序回执和实际文件为准。


In [ ]:
# @title ④ 查看程序回执、模型解释和图表
lab.show_results(run_output)


## ⑤ 下载与保存

下载本轮/全部结果ZIP、SKILL.md、Notebook源码及完整模块包。保留含实际运行输出的Notebook，请使用“文件→保存”，再用“文件→下载→下载.ipynb”。


In [ ]:
# @title ⑤ 保存与下载
lab.show_downloads(run_output)
